In [0]:
%sql

-- select count(*) from zara_lakehouse.retail_silver.dim_discount_silver
--select count(*) from zara_lakehouse.retail_silver.dim_products_sold_silver
--select count(*) from zara_lakehouse.retail_silver.dim_product_information_silver

--select count(*) from zara_lakehouse.retail_quarantine.dim_discount_qt
--select count(*) from zara_lakehouse.retail_quarantine.dim_products_sold_qt
--select count(*) from zara_lakehouse.retail_quarantine.dim_product_information_qt

-- select * from zara_lakehouse.retail_validation.dim_discount_dq
-- select * from zara_lakehouse.retail_validation.dim_products_sold_dq
select * from zara_lakehouse.retail_validation.dim_product_information_dq

-- select count(*) from zara_lakehouse.retail_silver.fact_sales_silver




In [0]:
%sql
select ps.date, ps.product_id, ps.city, ps.pieces_sold, p.price, d.discount 

from   zara_lakehouse.retail_silver.dim_products_sold_silver ps 
left join zara_lakehouse.retail_silver.dim_discount_silver d
on ps.product_id = d.product_id AND ps.city = d.city
left join zara_lakehouse.retail_silver.dim_product_information_silver p
on ps.product_id = p.product_id;


## Total Sales Revenue using discount 

In [0]:
%sql
WITH sales_enriched AS (
    SELECT 
        ps.date, 
        ps.product_id, 
        ps.city, 
        ps.pieces_sold, 
        p.price, 
        COALESCE(d.discount, 0) AS discount 
    FROM zara_lakehouse.retail_silver.dim_products_sold_silver ps 
    LEFT JOIN zara_lakehouse.retail_silver.dim_discount_silver d
        ON ps.product_id = d.product_id 
       AND ps.city = d.city
    LEFT JOIN zara_lakehouse.retail_silver.dim_product_information_silver p
        ON ps.product_id = p.product_id
)

SELECT 
    date, 
    product_id, 
    city, 

    SUM(pieces_sold) AS total_pieces_sold,
    SUM(pieces_sold * price * (1 - discount)) AS total_revenue
FROM sales_enriched
GROUP BY date, product_id, city;


In [0]:
%sql
WITH woman_winterdress_sales AS (
select 
pi.product_id,
pi.price,
ps.city, 
ps.date, 
ps.pieces_sold
from zara_lakehouse.retail_silver.dim_product_information_silver pi
 LEFT JOIN zara_lakehouse.retail_silver.dim_products_sold_silver ps
    on pi.product_id = ps.product_id
where pi.section = 'WOMAN' and pi.Season = 'Winter')
select 
ws.date,
ws.product_id,
price,
sum(pieces_sold) as total_pieces_sold,
d.discount
from woman_winterdress_sales
LEFT JOIN zara_lakehouse.retail_silver.dim_discount_silver d
GROUP BY date, product_id, city;


In [0]:
%sql
select MIN(date) as starting_date,
MAX(date) as ending_date
 from zara_lakehouse.retail_silver.dim_products_sold_silver

In [0]:
%sql
SELECT COUNT(date) FROM zara_lakehouse.retail_silver.dim_products_sold_silver;

In [0]:
%sql
WITH fact_sales AS (
SELECT 
    ps.product_iD,
    ps.city,
    ps.date,
    ps.pieces_sold,

    -- bring price from product table
    pi.price AS unit_price,

    -- revenue
    (ps.pieces_sold * pi.price) AS revenue,

    -- bring discount (LEFT JOIN because not all cities may exist)
    COALESCE(d.discount, 0) AS discount,

    -- discount amount
    (ps.pieces_sold * pi.price * COALESCE(d.discount, 0)) AS discount_amount,

    -- net revenue
    (ps.pieces_Sold * pi.price * (1 - COALESCE(d.discount, 0))) AS net_revenue

FROM zara_lakehouse.retail_silver.dim_products_sold_silver ps

JOIN zara_lakehouse.retail_silver.dim_product_information_silver pi 
    ON ps.Product_ID = pi.Product_ID

LEFT JOIN zara_lakehouse.retail_silver.dim_discount_silver d
    ON ps.Product_ID = d.Product_ID
    AND ps.City = d.City)
    select count(*) from fact_sales


--SELECT Product_ID, City, date, COUNT(*)
--FROM Fact_Sales
--GROUP BY Product_ID, City, date
--HAVING COUNT(*) > 1;